# Discover and download a Data Registry asset

This notebook accompanies Step 3 of the RHOAI 3.6 Data Registry walkthrough. Run it as an RHOAI user with project `admin` permissions in the selected project. It reads the fictional FAQ registered in Step 2, downloads it through the workbench's attached connection, verifies its checksum, and creates a second catalog entry through the API.

**Authentication:** provide your own OpenShift user token at the hidden prompt. The workbench ServiceAccount token is not used. The notebook does not print the user token or S3 credentials. Leave the token placeholder blank: JupyterLab can autosave a token pasted into a code cell.

Each `assert` below is a verification check. A failed check stops the walkthrough with a message instead of claiming success.


In [ ]:
# Install the two client libraries in this notebook's Python kernel.
# This uses the package index configured for the workbench.
%pip install --quiet --disable-pip-version-check "requests>=2.31,<3" "boto3>=1.34,<2"

## 1. Configure the example and check the runtime

Change `PROJECT` if Step 2 used a different project. The other defaults match the synthetic FAQ in the guide. Leave `USER_TOKEN` empty and enter your own token at the hidden prompt. Do not save a token in a notebook cell.

The setup cell installs `requests` and `boto3` in this notebook's Python kernel. The next cell imports the modules used throughout the walkthrough.

The connection attached in the workbench form supplies the five S3 settings. This preflight checks their **presence**, without showing values.


In [ ]:
import hashlib
import os
from getpass import getpass
from pathlib import Path
from urllib.parse import urlparse

import boto3
import requests
from botocore.config import Config
from botocore.exceptions import BotoCoreError, ClientError

# Public example names and the cluster Service URL.
PROJECT = "data-registry-example"
COLLECTION = "customer-support"
ASSET_NAME = "customer-support-faq"
CONNECTION_NAME = "data-registry-storage"
REGISTRY_URL = "https://feast-data-registry-data-registry.rhoai-data-registry.svc:443"

S3_PREFIX = "examples/customer-support/"
EXPECTED_KEY = S3_PREFIX + "customer-support-faq.md"
EXPECTED_SHA256 = "e5be400a6740cb71a0ae93606a194f6cfcde5db09460b8c88c6e50afb0e57f78"

# User token placeholder. The prompt keeps the token out of the saved notebook.
USER_TOKEN = ""  # Your user token placeholder; leave blank for the hidden prompt.
USER_TOKEN = USER_TOKEN or os.environ.get("REGISTRY_USER_TOKEN", "")
if not USER_TOKEN:
    USER_TOKEN = getpass("OpenShift user token: ").strip()

# Verification: fail early if the user token or attached connection is missing.
assert USER_TOKEN, "Provide your own OpenShift user token."
assert urlparse(REGISTRY_URL).scheme == "https", "Use the registry HTTPS Service."

required_s3 = (
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "AWS_S3_ENDPOINT",
    "AWS_DEFAULT_REGION",
    "AWS_S3_BUCKET",
)
S3_CONFIG = {name: os.environ.get(name) for name in required_s3}
missing = [name for name, value in S3_CONFIG.items() if not value]
assert not missing, (
    f"Attach {CONNECTION_NAME} and restart the workbench; missing: {missing}"
)

print("Project:", PROJECT)
print("Registry Service:", REGISTRY_URL)
print("Connection setting names present:", ", ".join(required_s3))

## 2. Create a certificate-verified API helper

OpenShift projects its **public** service CA into the workbench automatically. The helper sends your user token only to the registry's HTTPS Service and validates its serving certificate. Errors show the request path and status, never the token, headers, or response body.


In [ ]:
SERVICE_CA = Path("/var/run/secrets/kubernetes.io/serviceaccount/service-ca.crt")

# Verification: the workbench must have the public CA to validate HTTPS.
assert SERVICE_CA.is_file() and SERVICE_CA.stat().st_size, (
    "The public service CA is missing from this workbench."
)

http = requests.Session()
http.trust_env = False  # In-cluster Service calls should not use a workstation proxy.
http.headers.update({
    "Authorization": f"Bearer {USER_TOKEN}",
    "Accept": "application/json",
})


def api_request(method, path, *, body=None, authenticated=True, expected=(200,)):
    session = http if authenticated else requests.Session()
    session.trust_env = False

    try:
        response = session.request(
            method,
            REGISTRY_URL + path,
            json=body,
            verify=str(SERVICE_CA),
            timeout=20,
        )
    finally:
        if not authenticated:
            session.close()

    if response.status_code not in expected:
        raise RuntimeError(
            f"{method} {path}: HTTP {response.status_code}; expected {expected}."
        )
    if response.status_code in (204, 401, 403, 409):
        return response.status_code
    return response.json()


print("User-authenticated, certificate-verified API client is ready.")

## 3. Check authentication and list collections

The unauthenticated discovery call should return 401. Your user token should then allow project discovery and show `customer-support` among the collection names. These checks separate TLS reachability from authorization.


In [ ]:
anonymous_status = api_request(
    "GET", "/v1/config", authenticated=False, expected=(401,)
)
print("Anonymous discovery HTTP:", anonymous_status)

config = api_request("GET", f"/v1/{PROJECT}/config")
print("Authenticated project config keys:", sorted(config))

collections_path = f"/v1/{PROJECT}/namespaces"
collections = api_request("GET", collections_path)
collection_names = [parts[0] for parts in collections["namespaces"]]

# Verification: find the exact Step 2 collection, regardless of list order.
assert COLLECTION in collection_names, f"Collection {COLLECTION!r} was not found."
print("Collections:", collection_names)

## 4. Read the collection and asset metadata

The FAQ is a **volume** because it represents unstructured documents. Select it by exact name and check its format, S3 prefix, and typed connection reference. A registry response describes the data; it does not return S3 credentials or document bytes.


In [ ]:
collection_path = f"{collections_path}/{COLLECTION}"
collection = api_request("GET", collection_path)
assert collection["namespace"] == [COLLECTION]
print("Collection:", collection["namespace"][0])
print("Description:", collection.get("properties", {}).get("description", ""))

volumes_path = f"{collection_path}/volumes"
volumes = api_request("GET", volumes_path)["volumes"]
matching = [item for item in volumes if item.get("name") == ASSET_NAME]
assert len(matching) == 1, f"Expected one {ASSET_NAME!r}; found {len(matching)}."

asset = api_request("GET", f"{volumes_path}/{ASSET_NAME}")

# Verification: compare each identity and location field with Step 2.
assert asset["name"] == ASSET_NAME
assert asset["format"] == "documents"
assert asset["storage_location"] == f"s3://{S3_CONFIG['AWS_S3_BUCKET']}/{S3_PREFIX}"
assert asset["connection_ref"] == {
    "type": "secret",
    "secret_name": CONNECTION_NAME,
}

print("Asset:", asset["name"])
print("Format:", asset["format"])
print("S3 path:", asset["storage_location"])
print("Connection reference:", asset["connection_ref"]["secret_name"])

## 5. List objects through the attached connection

The workbench form supplies the connection settings as environment variables. The S3 client uses them explicitly, so it cannot silently fall back to another AWS identity. The internal SeaweedFS endpoint uses port 8333 and HTTP within the cluster; use HTTPS for an external provider.


In [ ]:
endpoint = S3_CONFIG["AWS_S3_ENDPOINT"].rstrip("/")
parsed = urlparse(endpoint)

# Verification: never send these credentials to an unexpected HTTP endpoint.
assert parsed.scheme in ("http", "https") and parsed.hostname
assert not parsed.username and not parsed.password and not parsed.path
if parsed.scheme == "http":
    assert parsed.hostname.endswith((".svc", ".svc.cluster.local")), (
        "Use HTTP only for an internal cluster Service."
    )

bucket = S3_CONFIG["AWS_S3_BUCKET"]
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    region_name=S3_CONFIG["AWS_DEFAULT_REGION"],
    aws_access_key_id=S3_CONFIG["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=S3_CONFIG["AWS_SECRET_ACCESS_KEY"],
    aws_session_token=os.environ.get("AWS_SESSION_TOKEN"),
    config=Config(s3={"addressing_style": "path"}, retries={"max_attempts": 3}),
)

try:
    keys = []
    pages = s3.get_paginator("list_objects_v2").paginate(
        Bucket=bucket, Prefix=S3_PREFIX
    )
    for page in pages:
        keys.extend(item["Key"] for item in page.get("Contents", []))
except (BotoCoreError, ClientError):
    raise RuntimeError(
        "S3 list failed; check the connection and storage network."
    ) from None

# Verification: the list must contain the exact committed fixture.
assert EXPECTED_KEY in keys, f"Fixture {EXPECTED_KEY!r} was not found."
print("Object keys under the asset prefix:", keys)

## 6. Download and verify the document

Download the exact object to `downloads/` and compare its SHA-256 with the committed synthetic fixture. This verifies the object bytes separately from the registry metadata.


In [ ]:
download_dir = Path("downloads")
download_dir.mkdir(exist_ok=True)
local_file = download_dir / "customer-support-faq.md"

try:
    document = s3.get_object(Bucket=bucket, Key=EXPECTED_KEY)["Body"].read()
except (BotoCoreError, ClientError):
    raise RuntimeError(
        "S3 download failed; check the key and connection access."
    ) from None

actual_sha256 = hashlib.sha256(document).hexdigest()

# Verification: a different object must not count as a successful walkthrough.
assert actual_sha256 == EXPECTED_SHA256, "Downloaded bytes differ from the fixture."
local_file.write_bytes(document)

print("Saved:", local_file.resolve())
print("Bytes:", len(document))
print("SHA-256:", actual_sha256)

## 7. Register a second collection through the API

Create `customer-support-api` to show an API write with your user token. A marker identifies this notebook's entry on reruns. A conflicting name with different metadata stops the walkthrough instead of overwriting someone else's collection.


In [ ]:
API_COLLECTION = "customer-support-api"
API_ASSET = "customer-support-faq-api"
MARKER = "data-registry-api-walkthrough"

api_collection_path = f"{collections_path}/{API_COLLECTION}"
new_collection = {
    "namespace": [API_COLLECTION],
    "properties": {
        "description": "API-created collection for the Data Registry workbench walkthrough.",
        "example.source": MARKER,
    },
}

result = api_request("POST", collections_path, body=new_collection, expected=(200, 409))
confirmed_collection = api_request("GET", api_collection_path)

# Verification: both a new POST and a safe rerun must read back our metadata.
assert confirmed_collection["namespace"] == new_collection["namespace"]
for key, value in new_collection["properties"].items():
    assert confirmed_collection.get("properties", {}).get(key) == value, (
        f"Collection {API_COLLECTION!r} has different {key!r} metadata."
    )
print("API collection:", API_COLLECTION, "(reused)" if result == 409 else "(created)")

## 8. Register the API asset and read it back

This creates another **catalog entry** pointing to the same S3 prefix and attached connection; it does not upload an object. The collection appears in the URL. Send the five descriptive attributes as top-level POST fields; the API returns them inside `properties` on GET. Compare every supplied field with the persisted response, including the marker used to recognize a safe rerun.


In [ ]:
api_volumes_path = f"{api_collection_path}/volumes"
new_asset = {
    "name": API_ASSET,
    "format": "documents",
    "description": "API-created catalog entry for the fictional customer support FAQ.",
    "storage_location": f"s3://{bucket}/{S3_PREFIX}",
    "connection_ref": {"type": "secret", "secret_name": CONNECTION_NAME},
    # The POST accepts these attributes at the top level.
    "purpose": "Demonstrate API registration and verified document retrieval.",
    "license": "internal-use",
    "maturity": "experimental",
    "domain": "customer-support",
    "pii": "none",
    "labels": ["customer-support", "synthetic", "api-example"],
    "properties": {"example.source": MARKER},
}

result = api_request("POST", api_volumes_path, body=new_asset, expected=(200, 409))
confirmed = api_request("GET", f"{api_volumes_path}/{API_ASSET}")

# Verification: check every supplied scalar and the typed connection reference.
for field in (
    "name",
    "format",
    "description",
    "storage_location",
    "connection_ref",
):
    assert confirmed.get(field) == new_asset[field], (
        f"API asset {API_ASSET!r} has different {field!r} metadata."
    )
assert set(confirmed.get("labels", [])) == set(new_asset["labels"])

# Verification: GET returns the top-level POST attributes under properties.
for key in ("purpose", "license", "maturity", "domain", "pii"):
    assert confirmed.get("properties", {}).get(key) == new_asset[key], (
        f"API asset {API_ASSET!r} has different {key!r} metadata."
    )
assert confirmed.get("properties", {}).get("example.source") == MARKER, (
    f"API asset {API_ASSET!r} has a different source marker."
)

print("API asset:", API_ASSET, "(reused)" if result == 409 else "(created)")
print("Read-back verified in collection:", API_COLLECTION)

## 9. Summary

You have verified the user-authorized API path, the catalog's connection reference, the attached connection's S3 access, and the downloaded document's checksum. You also created and read back a second collection and asset. The workbench file and API entries remain available for inspection; this notebook deletes nothing.

Keep `USER_TOKEN` blank when saving or sharing the notebook.


In [ ]:
print("Data Registry API walkthrough complete")
print("Step 2 asset:", f"{COLLECTION}/{ASSET_NAME}")
print("API asset:", f"{API_COLLECTION}/{API_ASSET}")
print("Verified object:", f"s3://{bucket}/{EXPECTED_KEY}")
print("Downloaded file:", local_file.resolve())
print("SHA-256 verified:", actual_sha256 == EXPECTED_SHA256)